# 06: Institutional Model Explainability, SHAP Attribution & Feature Selection

This interactive research notebook demonstrates the **MDK Trading Oracle Explainability & Feature Selection Engine** across **Models 1, 2, and 3**.

---

### Core Objectives
1. **Pillar 1: Local Signal Explainability (Live $T+1$)**:
   - Decomposes the upcoming session's forecast into an exact **Additive Waterfall**:
     $$\hat{y} = \mathbb{E}[y] + \sum_{i=1}^M \phi_i$$
     where $\mathbb{E}[y]$ is the baseline expected market flow, and $\phi_i$ is the exact TL (or return %) contribution of feature $i$.
2. **Pillar 2: Semantic Microstructure Cluster Rollup**:
   - Instead of 45 raw features causing cognitive overload, features are automatically rolled up into **institutional clusters** (*Closing Momentum*, *Competitor Deltas*, *Macro Rates*, *Tertip Inventory*, etc.) to reveal where true alpha originates.
3. **Pillar 3: Data-Driven Feature Pruning & Collinearity Screening**:
   - Uses **Out-of-Sample Permutation Importance** and **Pairwise Correlation Screening** ($|r| \ge 0.85$) to flag redundant features and recommend updates for `config/features.yaml`.


## 1. Setup & Lakehouse Connection

We connect to the local DuckDB lakehouse strictly in **read-only mode** (`read_only=True`) to ensure zero lock contention with active pipeline runs.


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

from mdk_trading_oracle.core.db import DuckDBManager
from mdk_trading_oracle.models.features_config import FeatureSelector
from mdk_trading_oracle.models.day_start.forecaster import DayStartForecaster
from mdk_trading_oracle.models.sector_day_start.forecaster import SectorDayStartForecaster
from mdk_trading_oracle.models.stock_reaction.forecaster import StockReactionForecaster
from mdk_trading_oracle.explainability import (
    ModelExplainer,
    FeatureAuditor,
    plot_waterfall,
    plot_cluster_donut,
    format_markdown_card,
)

# Connect in read-only mode to prevent file lock conflicts
db = DuckDBManager(read_only=True)
print("Lakehouse connection established (read-only mode).")


Lakehouse connection established (read-only mode).


## 2. Model 1 (Macro Day-Start): Live $T+1$ Signal Decomposition

Here we generate the live upcoming forecast for **Bank of America (MLB)** and decompose the predicted net flow into its exact additive catalysts and headwinds.


In [2]:
# Instantiate forecaster with read-only database connection
forecaster_m1 = DayStartForecaster(db=db)

# Compute live forecast for upcoming session T+1
res_m1 = forecaster_m1.forecast_next_day()

# Extract local explanation
exp_m1 = forecaster_m1.explain_forecast()

# Display formatted Markdown executive card
display(Markdown(format_markdown_card(exp_m1)))


[09/06/26 21:13:53] INFO     Extracting Day-Start 7 Feature Clusters for broker 'MLB' (lookback_months=12)...

                    INFO     Extracted 20 historical daily observations with 48 features.

[09/06/26 21:13:53] INFO     Running DayStartModelArena Walk-Forward Tournament (eval_window_days=20,              
                             min_burn_in=5, active_features=45)...

[09/06/26 21:13:59] INFO     🏆 Model Arena Champion Crowned: 'Bayesian Ridge Probabilistic' (Out-of-Sample Hit    
                             Rate: 86.7%, 90% PICP: 80.0%, RMSE: 506.89M TL)

                    INFO     Fitting Champion Model 'day_start_bayesian_ridge' on 20 historical daily sessions with
                             45 active feature(s)...

...

[09/06/26 21:14:08] INFO     Extracted 20 historical daily observations with 48 features.

[09/06/26 21:14:08] INFO     Running DayStartModelArena Walk-Forward Tournament (eval_window_days=20,              
                             min_burn_in=5, active_features=45)...

[09/06/26 21:14:11] INFO     🏆 Model Arena Champion Crowned: 'Bayesian Ridge Probabilistic' (Out-of-Sample Hit    
                             Rate: 86.7%, 90% PICP: 80.0%, RMSE: 506.89M TL)

                    INFO     Fitting Champion Model 'day_start_bayesian_ridge' on 20 historical daily sessions with
                             45 active feature(s)...

### Forecast Attribution Breakdown: MLB (-184.34M TL)
- **Expected Market Baseline (E[y])**: `-214.50M TL`
- **Net Forecast**: `-184.34M TL`

#### Top Catalysts (Bullish Drivers)
| Feature | Semantic Cluster | Value | Attribution |
| :--- | :--- | :---: | :---: |
| `feat_bofa_cum_net_flow_5d_tl` | inventory_saturation | -5,260,775,654.91 | **+53.49M TL** |
| `feat_bofa_vs_top5_inventory_delta_tl` | tertip_inventory | -3,366,409,415.04 | **+17.12M TL** |
| `feat_bofa_prev_day_turnover_tl` | inventory_saturation | 35,006,620,733.42 | **+12.42M TL** |

#### Top Headwinds (Bearish Drag)
| Feature | Semantic Cluster | Value | Attribution |
| :--- | :--- | :---: | :---: |
| `feat_bofa_vs_top5_w4_flow_delta_tl` | competitor_deltas | 4,493,601,352.35 | **-26.51M TL** |
| `feat_bofa_net_open_inventory_tl` | tertip_inventory | 6,313,945,634.57 | **-18.46M TL** |
| `feat_top5_domestic_w4_net_flow_tl` | competitor_deltas | -1,267,835,541.98 | **-5.78M TL** |

#### Microstructure Cluster Rollup
| Cluster | Net Contribution | Share (%) | Top Driver |
| :--- | :---: | :---: | :--- |
| **inventory_saturation** | +63.07M TL | 48.4% | `feat_bofa_cum_net_flow_5d_tl` |
| **tertip_inventory** | -1.24M TL | 25.1% | `feat_bofa_net_open_inventory_tl` |
| **competitor_deltas** | -31.78M TL | 23.1% | `feat_bofa_vs_top5_w4_flow_delta_tl` |
| **sector_flows** | +0.12M TL | 3.4% | `feat_top5_banking_flow_prev_day` |
| **closing_momentum** | +0.00M TL | 0.0% | `feat_bofa_w4_net_flow_tl` |
| **calendar_dynamics** | +0.00M TL | 0.0% | `day_of_week` |
| **cost_basis_pnl** | -0.00M TL | 0.0% | `feat_prev_day_close_vs_vwap_spread_pct` |
| **macro_rates** | -0.00M TL | 0.0% | `feat_macro_rate_shock_decay` |
| **institutional_hegemony** | +0.00M TL | 0.0% | `feat_institutional_hegemony_share` |
| **benchmark_index** | -0.00M TL | 0.0% | `feat_bist30_prev_day_intraday_return_pct` |

### Interactive Prediction Waterfall Chart

The waterfall shows the exact step-by-step path:
- **Baseline Expected Market Value $\mathbb{E}[y]$**: What BofA typically does on an average morning.
- **Green Bars (Catalysts)**: Market signals that pushed BofA's predicted flow higher.
- **Red Bars (Headwinds)**: Signals that exerted downward drag on BofA's predicted flow.
- **Blue Bar (Final Forecast $\hat{y}$)**: The resulting predicted opening net flow.


In [3]:
fig_waterfall_m1 = plot_waterfall(
    exp_m1,
    max_display=8,
    title=f"Model 1: BofA Day-Start Flow Decomposition ({res_m1.forecast_date})",
)
fig_waterfall_m1.show()


## 3. Global Microstructure Cluster Alpha Share

Across all historical sessions, which institutional feature clusters drive the majority of the model's predictive power?


In [4]:
# Compute cross-session global feature importance
global_exp_m1 = forecaster_m1.explain_global()

# Display Donut Chart showing cluster share %
fig_donut_m1 = plot_cluster_donut(
    global_exp_m1,
    title="Macro Day-Start: Microstructure Cluster Alpha Share (%)",
)
fig_donut_m1.show()

# Display tabular ranking
display(Markdown("### Semantic Microstructure Cluster Scoreboard"))
display(global_exp_m1.cluster_importance_df)


[09/06/26 21:14:14] INFO     Extracting Day-Start 7 Feature Clusters for broker 'MLB' (lookback_months=12)...

                    INFO     Extracted 20 historical daily observations with 48 features.

[09/06/26 21:14:14] INFO     Running DayStartModelArena Walk-Forward Tournament (eval_window_days=20,              
                             min_burn_in=5, active_features=45)...

[09/06/26 21:14:18] INFO     🏆 Model Arena Champion Crowned: 'Bayesian Ridge Probabilistic' (Out-of-Sample Hit    
                             Rate: 86.7%, 90% PICP: 80.0%, RMSE: 506.89M TL)

                    INFO     Fitting Champion Model 'day_start_bayesian_ridge' on 20 historical daily sessions with
                             45 active feature(s)...

...

[09/06/26 21:14:19] INFO     Running DayStartModelArena Walk-Forward Tournament (eval_window_days=20,              
                             min_burn_in=5, active_features=45)...

[09/06/26 21:14:23] INFO     🏆 Model Arena Champion Crowned: 'Bayesian Ridge Probabilistic' (Out-of-Sample Hit    
                             Rate: 86.7%, 90% PICP: 80.0%, RMSE: 506.89M TL)

                    INFO     Fitting Champion Model 'day_start_bayesian_ridge' on 20 historical daily sessions with
                             45 active feature(s)...

### Semantic Microstructure Cluster Scoreboard

,cluster_name,total_abs_attribution,relative_importance_pct,feature_count,top_feature,rank
0,tertip_inventory,5.256542e+07,4.128618e+01,6,feat_bofa_net_open_inventory_tl,1
1,inventory_saturation,4.403188e+07,3.458373e+01,6,feat_bofa_prev_day_turnover_tl,2
2,competitor_deltas,2.340516e+07,1.838299e+01,5,feat_bofa_vs_top5_w4_flow_delta_tl,3
3,sector_flows,7.317188e+06,5.747100e+00,8,feat_top5_banking_flow_prev_day,4
4,calendar_dynamics,1.199983e-02,9.424961e-09,3,day_of_week,5
5,closing_momentum,5.620299e-03,4.414322e-09,3,feat_bofa_w4_turnover_tl,6
6,cost_basis_pnl,8.646862e-06,6.791459e-12,2,feat_prev_day_close_vs_vwap_spread_pct,7
7,macro_rates,3.963096e-13,3.112714e-19,4,feat_macro_rate_shock_decay,8
8,benchmark_index,6.122028e-15,4.808392e-21,6,feat_bist30_cum_return_5d,9
9,institutional_hegemony,1.717211e-15,1.348740e-21,2,feat_institutional_hegemony_share,10


## 4. Model 2 (Sector Day-Start): Sector-Level Allocation Decomposition

Select any tracked BIST sector to inspect what microstructure features drove BofA's predicted capital allocation for that specific sector.


In [6]:
forecaster_m2 = SectorDayStartForecaster(db=db)

# Tracked liquid sectors
available_sectors = ["Banking", "Holding", "Transportation", "Energy & Refining", "Defense & Tech", "Retail Trade"]

def inspect_sector_explanation(sector_name):
    print(f"Generating live forecast & attribution for sector: {sector_name}...")
    exp_sec = forecaster_m2.explain_sector_forecast(sector=sector_name)
    if exp_sec:
        display(Markdown(format_markdown_card(exp_sec)))
        fig = plot_waterfall(exp_sec, max_display=6, title=f"BofA Day-Start Allocation: {sector_name}")
        fig.show()
    else:
        print(f"No explanation available for {sector_name}.")

# Interactive dropdown
dropdown = widgets.Dropdown(
    options=available_sectors,
    value="Banking",
    description="Sector:",
    disabled=False,
)
widgets.interact(inspect_sector_explanation, sector_name=dropdown);


interactive(children=(Dropdown(description='Sector:', options=('Banking', 'Holding', 'Transportation', 'Energy…

## 5. Model 3 (Stock Intraday Reaction): Equity Return % Decomposition

Select an individual BIST 30 equity and reaction window ($W_2, W_3, W_5$) to inspect the microstructure drivers behind the forecasted return percentage.


In [8]:
tracked_symbols = ["THYAO", "AKBNK", "GARAN", "EREGL", "TUPRS", "BIMAS", "ASELS", "KCHOL", "ISCTR", "YKBNK"]
window_options = [("Window 2: First Reaction (10:30-11:30)", "w2"),
                  ("Window 3: Midday Followup (11:30-14:30)", "w3"),
                  ("Window 5: Closing Session (16:00-18:15)", "w5")]

def inspect_stock_reaction(symbol, window_tuple):
    window_key = window_tuple
    print(f"Analyzing {symbol} ({window_key})...")
    forecaster_sr = StockReactionForecaster(symbol=symbol, window=window_key, db=db)
    res_sr = forecaster_sr.forecast_next_window(replace_active=False)
    
    if res_sr and res_sr.explanation:
        from mdk_trading_oracle.explainability.types import LocalExplanation, FeatureAttribution, ClusterAttribution
        d = res_sr.explanation
        exp = LocalExplanation(
            model_name=d["model_name"],
            model_version=d["model_version"],
            target_broker_or_symbol=d["target_broker_or_symbol"],
            base_value=d["base_value"],
            predicted_value=d["predicted_value"],
            unit=d.get("unit", "%"),
            top_positive_drivers=[FeatureAttribution(**item) for item in d.get("top_positive_drivers", [])],
            top_negative_drivers=[FeatureAttribution(**item) for item in d.get("top_negative_drivers", [])],
            cluster_attributions=[
                ClusterAttribution(
                    cluster_name=k,
                    total_attribution=v["total_attribution"],
                    total_abs_attribution=abs(v["total_attribution"]),
                    percentage_share=v["percentage_share"],
                    feature_count=1,
                    top_feature=v.get("top_feature", "")
                )
                for k, v in d.get("cluster_breakdown", {}).items()
            ]
        )
        display(Markdown(format_markdown_card(exp)))
        fig = plot_waterfall(exp, max_display=7, title=f"Stock Reaction Return % Waterfall: {symbol} ({window_key})")
        fig.show()
    else:
        print(f"Could not compute forecast/explanation for {symbol} ({window_key}).")

symbol_dd = widgets.Dropdown(options=tracked_symbols, value="THYAO", description="Symbol:")
window_dd = widgets.Dropdown(options=window_options, value="w2", description="Window:")
widgets.interact(inspect_stock_reaction, symbol=symbol_dd, window_tuple=window_dd);


interactive(children=(Dropdown(description='Symbol:', options=('THYAO', 'AKBNK', 'GARAN', 'EREGL', 'TUPRS', 'B…

## 6. Systematic Feature Selection & Pruning Audit

Run the **FeatureAuditor** to evaluate out-of-sample alpha contributions, detect collinear feature pairs ($|r| \ge 0.85$), and identify prune candidates.


In [9]:
# Run Feature Selection Audit on Model 1
report = forecaster_m1.audit_features(collinearity_threshold=0.85)

display(Markdown(f"""
### Feature Audit Summary: `{report.model_name}`
- **Evaluated Historical Sessions**: {report.evaluated_sessions}
- **Active Features Evaluated**: {report.total_features}
- **Collinear Pairs Detected**: {len(report.collinear_pairs)}
- **Prune Candidates Recommended**: {len(report.prune_candidates)}
"""))

# Top 10 Alpha Drivers
if report.top_drivers:
    display(Markdown("#### Top 10 Out-of-Sample Alpha Drivers"))
    display(pd.DataFrame(report.top_drivers))

# Collinear Pairs
if report.collinear_pairs:
    display(Markdown("#### Collinear Feature Redundancies (|r| >= 0.85)"))
    display(pd.DataFrame(report.collinear_pairs))

# Prune Candidates Table
if report.prune_candidates:
    display(Markdown("#### Recommended Features to Exclude"))
    display(pd.DataFrame(report.prune_candidates))

# Generated YAML Snippet
if report.recommended_features_yaml:
    display(Markdown("#### Recommended `config/features.yaml` Snippet"))
    print(report.recommended_features_yaml)


[09/06/26 21:19:41] INFO     Extracting Day-Start 7 Feature Clusters for broker 'MLB' (lookback_months=12)...

[09/06/26 21:19:42] INFO     Extracted 20 historical daily observations with 48 features.

[09/06/26 21:19:42] INFO     Running DayStartModelArena Walk-Forward Tournament (eval_window_days=20,              
                             min_burn_in=5, active_features=45)...

[09/06/26 21:19:45] INFO     🏆 Model Arena Champion Crowned: 'Bayesian Ridge Probabilistic' (Out-of-Sample Hit    
                             Rate: 86.7%, 90% PICP: 80.0%, RMSE: 506.89M TL)

                    INFO     Fitting Champion Model 'day_start_bayesian_ridge' on 20 historical daily sessions with
                             45 active feature(s)...

...

,feature_a,feature_b,correlation,cluster_a,cluster_b
0,feat_bofa_prev_day_net_flow_tl,feat_bofa_flow_zscore_20d,0.9711,inventory_saturation,inventory_saturation
1,feat_market_avg_return_pct,feat_bist30_prev_day_intraday_return_pct,0.9599,sector_flows,benchmark_index
2,feat_bofa_w4_net_flow_tl,feat_bofa_vs_top5_w4_flow_delta_tl,0.9526,closing_momentum,competitor_deltas
3,feat_bofa_prev_day_net_flow_tl,feat_bofa_vs_top5_total_flow_delta_tl,0.9343,inventory_saturation,competitor_deltas
4,feat_bofa_cost_basis_spread_20d_pct,feat_macro_rate_shock_decay,0.9203,cost_basis_pnl,macro_rates
5,feat_bofa_flow_zscore_20d,feat_bofa_vs_top5_total_flow_delta_tl,0.9114,inventory_saturation,competitor_deltas
6,feat_bofa_prev_day_net_flow_tl,feat_bist30_prev_day_intraday_return_pct,0.8989,inventory_saturation,benchmark_index
7,feat_bofa_prev_day_turnover_tl,feat_market_avg_range_pct,0.8879,inventory_saturation,sector_flows
8,feat_top5_domestic_w4_net_flow_tl,feat_bofa_vs_top5_w4_flow_delta_tl,0.8856,competitor_deltas,competitor_deltas
9,feat_bofa_vs_top5_total_flow_delta_tl,feat_bist30_prev_day_intraday_return_pct,0.8822,competitor_deltas,benchmark_index


#### Recommended Features to Exclude

,feature_name,cluster_name,permutation_score_drop,mean_abs_shap,reason
0,feat_w4_flow_acceleration_ratio,closing_momentum,0.000000e+00,0.000000e+00,Negative/Zero out-of-sample alpha contribution
1,feat_bofa_prev_day_net_flow_tl,inventory_saturation,-1.561834e+05,1.468082e+06,Negative/Zero out-of-sample alpha contribution
2,feat_bofa_prev_day_market_share,inventory_saturation,0.000000e+00,0.000000e+00,Negative/Zero out-of-sample alpha contribution
3,feat_bofa_prev_day_turnover_rank,inventory_saturation,0.000000e+00,0.000000e+00,Negative/Zero out-of-sample alpha contribution
4,feat_bofa_cum_net_flow_5d_tl,inventory_saturation,-4.010701e+05,1.922646e+07,Negative/Zero out-of-sample alpha contribution
5,feat_bofa_flow_zscore_20d,inventory_saturation,0.000000e+00,0.000000e+00,Collinear redundant with feat_bofa_prev_day_ne...
6,feat_bofa_cost_basis_spread_20d_pct,cost_basis_pnl,0.000000e+00,0.000000e+00,Negative/Zero out-of-sample alpha contribution
7,feat_prev_day_close_vs_vwap_spread_pct,cost_basis_pnl,0.000000e+00,0.000000e+00,Negative/Zero out-of-sample alpha contribution
8,feat_bofa_vs_top5_w4_flow_delta_tl,competitor_deltas,-8.413907e+04,1.527468e+07,Collinear redundant with feat_bofa_w4_net_flow...
9,feat_bofa_vs_top5_total_flow_delta_tl,competitor_deltas,4.339810e+06,0.000000e+00,Collinear redundant with feat_bofa_prev_day_ne...


#### Recommended `config/features.yaml` Snippet

# Recommended updates for config/features.yaml (day_start):
day_start:
  exclude_features:
    - feat_w4_flow_acceleration_ratio  # Negative/Zero out-of-sample alpha contribution
    - feat_bofa_prev_day_net_flow_tl  # Negative/Zero out-of-sample alpha contribution
    - feat_bofa_prev_day_market_share  # Negative/Zero out-of-sample alpha contribution
    - feat_bofa_prev_day_turnover_rank  # Negative/Zero out-of-sample alpha contribution
    - feat_bofa_cum_net_flow_5d_tl  # Negative/Zero out-of-sample alpha contribution
    - feat_bofa_flow_zscore_20d  # Collinear redundant with feat_bofa_prev_day_net_flow_tl (r >= 0.85)
    - feat_bofa_cost_basis_spread_20d_pct  # Negative/Zero out-of-sample alpha contribution
    - feat_prev_day_close_vs_vwap_spread_pct  # Negative/Zero out-of-sample alpha contribution
    - feat_bofa_vs_top5_w4_flow_delta_tl  # Collinear redundant with feat_bofa_w4_net_flow_tl (r >= 0.85)
    - feat_bofa_vs_top5_total_flow_delta_tl  # Collinear redundant with feat